# QuDock demo: quantum kernels for BACE-1 inhibitor screening

This notebook walks through the QuDock pipeline end to end on a small scale, so it runs in a
couple of minutes on a laptop:

1. load and split the MoleculeNet BACE dataset
2. turn molecules into qubit-ready features
3. build the ZZ feature-map quantum kernel and compare it with a classical RBF kernel
4. train a QSVC and classical baselines
5. look at the full benchmark, qubit sweep and noise study results
6. compare the kernel measured on a real IBM quantum computer with the ideal one

The full experiments are run from the command line (`qudock benchmark`, `qudock sweep`,
`qudock noise`, `qudock hardware`); their outputs live in `results/full/` and
`results/hardware/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from qudock.analysis import geometric_difference, kernel_target_alignment, off_diagonal_stats
from qudock.classical import make_classical, scores
from qudock.data import load_bace, scaffold_split
from qudock.features import FeaturePipeline
from qudock.quantum import QSVC, ZZKernel

RESULTS = Path("../results/full")

## 1. Data

BACE has 1,513 molecules labelled as β-secretase 1 inhibitors (1) or not (0). We use a
**scaffold split**: molecules sharing a Bemis–Murcko scaffold stay on the same side, so the test
set contains chemotypes the model has never seen. This is harder, and more realistic, than a
random split.

In [ ]:
df = load_bace("../data/cache")
split = scaffold_split(df["smiles"])
y = df["label"].values
print(f"{len(df)} molecules, {y.mean():.1%} inhibitors")
print(f"train {len(split.train)}  test {len(split.test)}")
df.head()

## 2. Features

Each molecule becomes a 2048-bit Morgan fingerprint, which is standardised and compressed with
PCA to one number per qubit, then rescaled to [0, π]. Everything is fit on the training set only.
To keep this notebook quick we use 4 qubits and a 300-molecule training subsample.

In [ ]:
rng = np.random.default_rng(0)
train = rng.choice(split.train, 300, replace=False)
test = split.test

pipe = FeaturePipeline(n_qubits=4)
X_train = pipe.fit_transform(df["smiles"].values[train])
X_test = pipe.transform(df["smiles"].values[test])
y_train, y_test = y[train], y[test]
print(X_train.shape, f"explained variance kept: {pipe.explained_variance:.1%}")

## 3. The quantum kernel

The ZZ feature map encodes a feature vector into a 4-qubit state. The kernel between two
molecules is the overlap of their states, $k(x, y) = |\langle\phi(x)|\phi(y)\rangle|^2$.

In [ ]:
kernel = ZZKernel(n_qubits=4, reps=2, bandwidth=0.5)
kernel.circuit.decompose().draw("mpl", fold=40)

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

order = np.argsort(y_train)[::-1][:60]  # 60 molecules, inhibitors first
K_q = kernel(X_train)
K_c = rbf_kernel(X_train, gamma=1 / (4 * X_train.var()))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, K, title in [(axes[0], K_q, "ZZ quantum kernel"), (axes[1], K_c, "RBF kernel")]:
    im = ax.imshow(K[np.ix_(order, order)], vmin=0, vmax=1, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

print(f"kernel-target alignment  quantum {kernel_target_alignment(K_q, y_train):.3f}"
      f"  RBF {kernel_target_alignment(K_c, y_train):.3f}")
print(f"geometric difference g(RBF || quantum) = {geometric_difference(K_c, K_q):.1f}")
print("quantum off-diagonal stats:", off_diagonal_stats(K_q))

## 4. Train and compare

QSVC is an ordinary SVM on the precomputed quantum kernel. We compare it with three classical
models on exactly the same 4 features, all tuned by cross-validation.

In [ ]:
from sklearn.metrics import roc_auc_score

aucs = {}
qsvc = QSVC(C=1, reps=2, bandwidth=0.5).fit(X_train, y_train)
aucs["QSVC (ZZ kernel)"] = roc_auc_score(y_test, qsvc.decision_function(X_test))
for name in ["svm_rbf", "random_forest", "logreg"]:
    model = make_classical(name, n_train=len(y_train)).fit(X_train, y_train)
    aucs[name] = roc_auc_score(y_test, scores(model, X_test))

pd.Series(aucs, name="test ROC-AUC").round(3).to_frame()

## 5. Full results

The figures below come from the complete runs in `results/full/` (all 1,210 training molecules,
several seeds).

### Sample efficiency

Test ROC-AUC as the training set grows from 25 to 1,210 molecules (6 qubits, 5 seeds).

In [ ]:
display(Image(RESULTS / "learning_curve.png"))
pd.read_csv(RESULTS / "qsvc_vs_classical.csv").round(3)

### Qubit sweep and kernel concentration

Left: ROC-AUC from 2 to 10 qubits. Right: as qubits grow, the off-diagonal kernel values
bunch together (the kernel *concentrates*), and a smaller bandwidth slows that down.

In [ ]:
display(Image(RESULTS / "qubit_sweep.png"))

### Noise study

Kernel entries estimated from measurement shots, with and without the noise model of IBM's
Torino device (simulated).

In [ ]:
path = RESULTS / "noise_study.png"
if path.exists():
    display(Image(path))
    noise = pd.read_csv(RESULTS / "noise_study.csv")
    display(noise[~noise["psd_repair"]].groupby(["reps", "noise", "shots"])
            [["rel_frobenius", "roc_auc"]].mean().round(3))
else:
    print("run `qudock noise --out results/full` to produce the noise study")

## 6. On a real IBM quantum computer

We ran the same compute-uncompute kernel circuits for 12 test molecules (6 inhibitors, 6 not) on
a real IBM device: 66 circuits with 1,024 shots each, in a single job. The same circuits were
also simulated with the FakeTorino noise model, using identical transpilation.

In [ ]:
import json

HARDWARE = Path("../results/hardware")
info = json.loads((HARDWARE / "hardware_run.json").read_text())
print(f"{info['backend']}: {info['n_circuits']} circuits x {info['shots']} shots, "
      f"{info['quantum_seconds']} s of quantum time")

saved = np.load(HARDWARE / "hardware_kernels.npz")
panels = [("exact_reps2", "exact"), ("faketorino_reps2", "FakeTorino simulation"),
          ("device_reps2", f"{info['backend']} (real device)")]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, (key, title) in zip(axes, panels):
    im = ax.imshow(saved[key], vmin=0, vmax=1, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

pd.read_csv(HARDWARE / "hardware_kernel.csv").round(3)

## Takeaways

* On this dataset the ZZ-kernel QSVC is competitive with an RBF SVM on the same compressed
  features, beats logistic regression, but does not beat Random Forest.
* The quantum kernel is geometrically different from the RBF kernel (large geometric difference)
  and, at a moderate bandwidth, aligns better with the labels, yet that does not turn into a
  better classifier here.
* Kernel concentration grows with qubit count; bandwidth tuning is essential.
* Device noise distorts the kernel more than shot noise does (8 to 13% error with simulated Torino
  noise that more shots cannot remove, against 1 to 6% from shots alone), yet test ROC-AUC
  barely moves: the noise mostly shrinks all kernel values together, which leaves the SVM's
  ranking of molecules largely intact.
* On the real ibm_marrakesh device the kernel was 17.5% off the ideal one, against 10.3%
  predicted by the FakeTorino simulation, yet its shape was preserved (correlation 0.996 with
  the exact kernel). Real hardware is noisier than the simulation, but the kernel survives.